In [12]:
import importlib.util
import sys
from datetime import date
import pandas as pd
import numpy as np
from typing import Dict, Literal, Optional
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.dates as mdates
import os
from pathlib import Path
from src.dataloading.epidataloader import EpiDataLoader
from src.dataloading.gnndataloader import GNNDataLoader


wissdaten_dir    = os.environ.get('TMPDIR') +'/wissdaten/'
data_env         = os.path.join(wissdaten_dir, 'ZKI-PH4/deschrijvers_wissdaten/data')

data_module      = module_path = data_env + "/__init__.py"
spec = importlib.util.spec_from_file_location("data", module_path)
module = importlib.util.module_from_spec(spec)
sys.modules["data"] = module
spec.loader.exec_module(module)

from src.utils.constants import *

epidata_nuts2 = GNNDataLoader('influenza', data_env, nuts_level='nuts2', min_date='2006-05-15',max_date='2020-06-01', include_population=True, periods = 12, prediction_horizon=1)
epidata_nuts2.add_time_features()
epidata_nuts2.log_transform_target()
epidata_nuts2.set_splits()
epidata_nuts2.normalize()
epidata_nuts2.add_lagged_features(lags = range(2,3))
epidata_nuts2.finalize()

epidata_nuts2_id  = epidata_nuts2.copy(deep=True).retrieve_graph(graphname = 'identity_graph').construct_dataloaders()
epidata_nuts2_bn  = epidata_nuts2.copy(deep=True).retrieve_graph(graphname = 'boolean_neighbors_self0').construct_dataloaders()
epidata_nuts2_gm1 = epidata_nuts2.copy(deep=True).retrieve_graph(graphname = 'gravity_model_dense').construct_dataloaders()
epidata_nuts2_gm2 = epidata_nuts2.copy(deep=True).retrieve_graph(graphname = 'gravity_model_sparse').construct_dataloaders()

GNN temporal windowing: extending data collection from 2020-06-01 to 2020-08-24 (+12 weeks)
GNN temporal windowing: extending data collection from 2020-06-01 to 2020-08-24 (+12 weeks)
GNN temporal windowing: extending data collection from 2020-06-01 to 2020-08-24 (+12 weeks)
GNN temporal windowing: extending data collection from 2020-06-01 to 2020-08-24 (+12 weeks)
GNN temporal windowing: extending data collection from 2020-06-01 to 2020-08-24 (+12 weeks)


In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GATConv, ChebConv, GINConv
from torch_geometric_temporal.nn.recurrent import DCRNN, TGCN, A3TGCN
import pandas as pd
import numpy as np
from typing import Optional, Tuple

from src.models._deepmodel import DeepModel


class TemporalGCN(nn.Module):
    """
    Temporal Graph Convolutional Network that properly uses graph structure
    across time steps for infectious disease forecasting.
    """
    def __init__(self, node_features: int, hidden_size: int = 64, num_layers: int = 2, 
                 dropout: float = 0.2, temporal_layers: int = 2, prediction_horizon: int = 1):
        super(TemporalGCN, self).__init__()
        
        self.node_features = node_features
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.temporal_layers = temporal_layers
        self.prediction_horizon = prediction_horizon
        
        # Spatial GCN layers
        self.spatial_convs = nn.ModuleList()
        self.spatial_convs.append(GCNConv(node_features, hidden_size))
        
        for _ in range(num_layers - 1):
            self.spatial_convs.append(GCNConv(hidden_size, hidden_size))
        
        # Temporal LSTM layers
        self.temporal_lstm = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=temporal_layers,
            dropout=dropout if temporal_layers > 1 else 0,
            batch_first=True
        )
        
        # Output projection
        self.output_proj = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            # nn.ReLU(),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, prediction_horizon)
        )
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x: torch.Tensor, edge_index: torch.Tensor, 
                edge_weight: Optional[torch.Tensor] = None, debug = False) -> torch.Tensor:
        """
        x: [num_nodes, node_features, time_steps]
        edge_index: [2, num_edges]
        edge_weight: [num_edges] (optional)
        """
        num_nodes, node_features, time_steps = x.shape
        
        # Process each time step through spatial GCN
        spatial_outputs = []
        
        for t in range(time_steps):
            # Get features for current time step
            x_t = x[:, :, t]  # [num_nodes, node_features]
            
            # Apply spatial GCN layers
            h = x_t
            for conv in self.spatial_convs:
                h = conv(h, edge_index, edge_weight)
                h = F.relu(h)
                h = self.dropout(h)
            
            spatial_outputs.append(h)
        
        # Stack spatial outputs along time dimension
        spatial_seq = torch.stack(spatial_outputs, dim=1)  # [num_nodes, time_steps, hidden_size]
        
        # Apply temporal LSTM
        lstm_out, _ = self.temporal_lstm(spatial_seq)  # [num_nodes, time_steps, hidden_size]
        
        # Always use the last hidden state for predictions
        last_hidden = lstm_out[:, -1, :]  # [num_nodes, hidden_size]
        output = self.output_proj(last_hidden)  # [num_nodes, prediction_horizon]

        return output  # [num_nodes, prediction_horizon]



class TGCNModel(DeepModel):
    """
    Purely spatial GCN model that does not use temporal axis.
    Useful to validate the use of graph-structure
    """
    def __init__(self, 
                 dataloader: GNNDataLoader, 
                 name: Optional[str] = None):
        super().__init__(dataloader, name=name)
        if not self.name:
            self.name = 'TGCN'

        self.model_color = '#4ECDC4'
        self.dataloader = dataloader
        self.config_info['model'] = 'tgcnmodel'

    def set_model_hparams(self, hidden_size: int = 64, num_layers: int = 2,
                         temporal_layers: int = 2, dropout: float = 0.2):
        self.model_hparams_set = True
        self.model = TemporalGCN(
            node_features=len(self.dataloader.feature_columns),
            hidden_size=hidden_size,
            num_layers=num_layers,
            temporal_layers=temporal_layers,
            dropout=dropout,
            prediction_horizon= self.dataloader.prediction_horizon
        ).to(self.device)
        
        model_hparams_config = {'hidden_size': hidden_size,
                                'num_layers' : num_layers,
                                'temporal_layers':temporal_layers,
                                'dropout':dropout}

        self.config_info['model_hparams'] = model_hparams_config
        self._state['model_initialized'] = True


        return self


In [13]:
epidata_nuts2_id.dataloader_main[0]

GraphDataLoaderEntry(x=[38, 4, 12], y=[38, 1], edge_index=[2, 38], edge_weight=[38])

In [ ]:
n_epochs = 100
lr       = 0.0001

ml2_id = TGCNModel(name = f'tgcn-identity_graph', dataloader=epidata_nuts2_id)
ml2_id.set_model_hparams()
ml2_id.set_global_hparams(lr = lr, n_epochs=n_epochs, scheduler_kwargs={'step_size':20, 'gamma':0.95}, min_delta = 0.00001, loss = 'mse')
ml2_id.run_snapshot(debug=True)
ml2_id.train(verbose=2)
ml2_id.forecast()
ml2_id.show_forecasts(dataset='test', target_h = 0)

ml2_bn = TGCNModel(name = f'tgcn-boolean_neighbors', dataloader=epidata_nuts2_bn)
ml2_bn.set_model_hparams()
ml2_bn.set_global_hparams(lr = lr, n_epochs=n_epochs, scheduler_kwargs={'step_size':20, 'gamma':0.95}, min_delta = 0.00001, loss = 'mse')
ml2_bn.run_snapshot(debug=True)
ml2_bn.train(verbose=2)
ml2_bn.forecast()
ml2_bn.show_forecasts(dataset='test', target_h = 0)

ml2_gm1 = TGCNModel(name = f'tgcn-gravity1', dataloader=epidata_nuts2_gm1)
ml2_gm1.set_model_hparams()
ml2_gm1.set_global_hparams(lr = lr, n_epochs=n_epochs, scheduler_kwargs={'step_size':20, 'gamma':0.95}, min_delta = 0.00001, loss = 'mse')
ml2_gm1.run_snapshot(debug=True)
ml2_gm1.train(verbose=2)
ml2_gm1.forecast()
ml2_gm1.show_forecasts(dataset='test', target_h = 0)

ml2_gm2 = TGCNModel(name = f'tgcn-gravity2', dataloader=epidata_nuts2_gm2)
ml2_gm2.set_model_hparams()
ml2_gm2.set_global_hparams(lr = lr, n_epochs=n_epochs, scheduler_kwargs={'step_size':20, 'gamma':0.95}, min_delta = 0.00001, loss = 'mse')
ml2_gm2.run_snapshot(debug=True)
ml2_gm2.train(verbose=2)
ml2_gm2.forecast()
ml2_gm2.show_forecasts(dataset='test', target_h = 0)


🧪 Snapshot 0 validation summary:
➡️  Predicted shape: torch.Size([38, 1])
➡️  Ground truth shape: torch.Size([38, 1])
✅ Snapshot ran successfully
Dataloader Snapshot: GraphDataLoaderEntry(x=[38, 4, 12], y=[38, 1], edge_index=[2, 38], edge_weight=[38])
Epoch 1 train loss: 0.6542, val loss: 1.0621 ✓ (new best)
Epoch 2 train loss: 0.4752, val loss: 0.6294 ✓ (new best)
